# Apache Iceberg com Apache Spark (PySpark)

## TechStore — Sistema de Vendas

Este notebook demonstra as operações fundamentais do **Apache Iceberg** integrado ao **Apache Spark**, usando o mesmo cenário da loja virtual **TechStore**, permitindo a comparação direta com o notebook Delta Lake.

### O que será demonstrado:
- Configuração do ambiente PySpark + Iceberg (via JAR Maven)
- Criação de namespace e tabelas Iceberg
- Geração de dados sintéticos com `Faker`
- **INSERT** — Escrita inicial de dados
- **UPDATE** — Atualização de registros via SQL
- **DELETE** — Remoção de registros via SQL
- **MERGE** — Upsert via `MERGE INTO`
- **Time Travel** — Consulta de snapshots históricos
- **Partition Evolution** — Mudança de particionamento sem reescrever dados
- **Manutenção de tabelas** — Compactação e expiração de snapshots

### Modelo de Dados (mesmo do Delta Lake)
```
clientes  ──┐
            ├─ pedidos ──┐
produtos  ──┘            └─ itens_pedido
```

---
## 1. Verificação do Ambiente

In [ ]:
import sys

print(f"Python: {sys.version}")

try:
    import pyspark
    print(f"PySpark: {pyspark.__version__}")
except ImportError:
    print("PySpark nao encontrado. Execute: uv sync")

try:
    from faker import Faker
    print("Faker: OK")
except ImportError:
    print("Faker nao encontrado. Execute: uv sync")

print("\nNOTA: O Apache Iceberg nao requer pacote Python adicional.")
print("O JAR necessario sera baixado automaticamente do Maven Central")
print("na primeira execucao (requer conexao com internet).")

---
## 2. Configuração da SparkSession com Apache Iceberg

> **Importante:** Na primeira execução, o Spark fará o download do JAR do Iceberg (~50MB) do Maven Central. Isso pode levar alguns minutos dependendo da conexão.

In [ ]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DoubleType, DateType, LongType
)

# Remover warehouse anterior para ambiente limpo
WAREHOUSE_PATH = "../warehouse/iceberg"
if os.path.exists(WAREHOUSE_PATH):
    shutil.rmtree(WAREHOUSE_PATH)
    print(f"Warehouse removido: {WAREHOUSE_PATH}")

ICEBERG_VERSION = "1.6.1"
SPARK_VERSION   = "3.5"
SCALA_VERSION   = "2.12"
ICEBERG_JAR = (
    f"org.apache.iceberg:"
    f"iceberg-spark-runtime-{SPARK_VERSION}_{SCALA_VERSION}:"
    f"{ICEBERG_VERSION}"
)

spark = (
    SparkSession.builder
    .appName("TechStore-Iceberg")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.jars.packages", ICEBERG_JAR)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.local.type", "hadoop")
    .config(
        "spark.sql.catalog.local.warehouse",
        os.path.abspath(WAREHOUSE_PATH).replace("\\", "/")
    )
    .config("spark.sql.defaultCatalog", "local")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"SparkSession iniciada com sucesso!")
print(f"Versao Spark : {spark.version}")
print(f"Iceberg JAR  : {ICEBERG_JAR}")
print(f"Warehouse    : {os.path.abspath(WAREHOUSE_PATH)}")
print(f"UI disponivel: http://localhost:4040")

---
## 3. Criação do Namespace e Tabelas Iceberg

In [ ]:
# Criar namespace (equivalente a schema/database)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.techstore")
print("Namespace 'local.techstore' criado.")

# Criar tabelas Iceberg via DDL SQL
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.techstore.clientes (
        cliente_id   INT    NOT NULL,
        nome         STRING NOT NULL,
        email        STRING NOT NULL,
        cidade       STRING,
        estado       STRING,
        data_cadastro DATE
    )
    USING iceberg
    PARTITIONED BY (estado)
""")
print("Tabela 'clientes' criada (particionada por estado).")

spark.sql("""
    CREATE TABLE IF NOT EXISTS local.techstore.produtos (
        produto_id INT    NOT NULL,
        nome       STRING NOT NULL,
        categoria  STRING NOT NULL,
        preco      DOUBLE NOT NULL,
        estoque    INT    NOT NULL
    )
    USING iceberg
    PARTITIONED BY (categoria)
""")
print("Tabela 'produtos' criada (particionada por categoria).")

spark.sql("""
    CREATE TABLE IF NOT EXISTS local.techstore.pedidos (
        pedido_id   INT    NOT NULL,
        cliente_id  INT    NOT NULL,
        data_pedido DATE   NOT NULL,
        status      STRING NOT NULL,
        valor_total DOUBLE NOT NULL
    )
    USING iceberg
    PARTITIONED BY (status, months(data_pedido))
""")
print("Tabela 'pedidos' criada (particionada por status e mes).")

spark.sql("""
    CREATE TABLE IF NOT EXISTS local.techstore.itens_pedido (
        item_id        INT    NOT NULL,
        pedido_id      INT    NOT NULL,
        produto_id     INT    NOT NULL,
        quantidade     INT    NOT NULL,
        preco_unitario DOUBLE NOT NULL
    )
    USING iceberg
""")
print("Tabela 'itens_pedido' criada.")

print("\nTabelas no namespace techstore:")
spark.sql("SHOW TABLES IN local.techstore").show()

---
## 4. Geração de Dados Sintéticos

In [ ]:
from faker import Faker
import random
from datetime import date, timedelta

fake = Faker('pt_BR')
Faker.seed(42)
random.seed(42)

ESTADOS = ['SC', 'SP', 'PR', 'RS', 'RJ', 'MG', 'BA', 'GO']
CIDADES = {
    'SC': ['Criciuma', 'Florianopolis', 'Blumenau', 'Joinville', 'Tubarao'],
    'SP': ['Sao Paulo', 'Campinas', 'Santos', 'Sorocaba'],
    'PR': ['Curitiba', 'Londrina', 'Maringa', 'Ponta Grossa'],
    'RS': ['Porto Alegre', 'Caxias do Sul', 'Pelotas'],
    'RJ': ['Rio de Janeiro', 'Niteroi', 'Petropolis'],
    'MG': ['Belo Horizonte', 'Uberlandia', 'Juiz de Fora'],
    'BA': ['Salvador', 'Feira de Santana', 'Vitoria da Conquista'],
    'GO': ['Goiania', 'Anapolis', 'Aparecida de Goiania'],
}

clientes_data = []
for i in range(1, 101):
    estado = random.choice(ESTADOS)
    cidade = random.choice(CIDADES[estado])
    clientes_data.append((
        i, fake.name(), f"cliente{i}@techstore.com",
        cidade, estado,
        fake.date_between(start_date=date(2022, 1, 1), end_date=date(2024, 12, 31)),
    ))

PRODUTOS_LIST = [
    ('Smartphone Samsung Galaxy S24','Smartphones',3299.99,45),
    ('iPhone 15 Pro','Smartphones',7999.99,20),
    ('Notebook Dell Inspiron 15','Notebooks',3899.99,30),
    ('MacBook Air M2','Notebooks',9999.99,15),
    ('Smart TV LG 55 4K','TVs',2799.99,25),
    ('Smart TV Samsung 65 QLED','TVs',4999.99,12),
    ('Tablet iPad Air','Tablets',4499.99,18),
    ('Tablet Samsung Galaxy Tab S9','Tablets',3199.99,22),
    ('Fone Sony WH-1000XM5','Acessorios',1899.99,60),
    ('Fone AirPods Pro 2','Acessorios',2299.99,35),
    ('Teclado Mecanico Keychron K2','Perifericos',599.99,50),
    ('Mouse Logitech MX Master 3','Perifericos',699.99,40),
    ('Monitor Dell 27 IPS','Monitores',1899.99,20),
    ('Monitor LG UltraWide 34','Monitores',2499.99,15),
    ('SSD Externo Samsung T7 1TB','Armazenamento',499.99,80),
    ('HD Externo Seagate 2TB','Armazenamento',349.99,65),
    ('Roteador TP-Link AX3000','Redes',499.99,55),
    ('Webcam Logitech C920','Perifericos',699.99,45),
    ('Impressora HP LaserJet','Impressoras',1299.99,10),
    ('Caixa de Som JBL Charge 5','Acessorios',999.99,70),
    ('Console PlayStation 5','Games',4299.99,8),
    ('Console Xbox Series X','Games',4199.99,10),
    ('Controle DualSense PS5','Games',499.99,40),
    ('Nintendo Switch OLED','Games',2899.99,18),
    ('Carregador MagSafe 20W','Acessorios',199.99,100),
    ('Capa iPhone 15 Silicone','Acessorios',99.99,200),
    ('Cabo USB-C Anker 2m','Acessorios',79.99,150),
    ('Hub USB-C 7-em-1','Perifericos',299.99,60),
    ('Suporte para Notebook','Acessorios',149.99,80),
    ('Webcam 4K Elgato Facecam','Perifericos',1299.99,20),
]

produtos_data = [(i+1, n, c, p, e) for i,(n,c,p,e) in enumerate(PRODUTOS_LIST)]

STATUS_OPCOES = ['PENDENTE', 'PROCESSANDO', 'ENVIADO', 'ENTREGUE', 'CANCELADO']
pedidos_data = []
for i in range(1, 201):
    cliente_id = random.randint(1, 100)
    data_pedido = fake.date_between(start_date=date(2024, 1, 1), end_date=date(2024, 12, 31))
    status = random.choices(STATUS_OPCOES, weights=[10,15,20,45,10])[0]
    valor_total = round(random.uniform(79.99, 15000.00), 2)
    pedidos_data.append((i, cliente_id, data_pedido, status, valor_total))

itens_data = []
item_id = 1
for pedido_id, _, _, _, _ in pedidos_data:
    n_itens = random.randint(1, 4)
    for prod in random.sample(produtos_data, n_itens):
        produto_id, _, _, preco, _ = prod
        qtd = random.randint(1, 3)
        itens_data.append((item_id, pedido_id, produto_id, qtd, preco))
        item_id += 1

print("Dados gerados:")
print(f"  Clientes     : {len(clientes_data):>5}")
print(f"  Produtos     : {len(produtos_data):>5}")
print(f"  Pedidos      : {len(pedidos_data):>5}")
print(f"  Itens pedido : {len(itens_data):>5}")

---
## 5. INSERT — Inserindo Dados nas Tabelas Iceberg

In [ ]:
schema_clientes = StructType([
    StructField("cliente_id",   IntegerType(), False),
    StructField("nome",         StringType(),  False),
    StructField("email",        StringType(),  False),
    StructField("cidade",       StringType(),  True),
    StructField("estado",       StringType(),  True),
    StructField("data_cadastro",DateType(),     True),
])

schema_produtos = StructType([
    StructField("produto_id",IntegerType(), False),
    StructField("nome",      StringType(),  False),
    StructField("categoria", StringType(),  False),
    StructField("preco",     DoubleType(),  False),
    StructField("estoque",   IntegerType(), False),
])

schema_pedidos = StructType([
    StructField("pedido_id",   IntegerType(), False),
    StructField("cliente_id",  IntegerType(), False),
    StructField("data_pedido", DateType(),     False),
    StructField("status",      StringType(),  False),
    StructField("valor_total", DoubleType(),  False),
])

schema_itens = StructType([
    StructField("item_id",       IntegerType(), False),
    StructField("pedido_id",     IntegerType(), False),
    StructField("produto_id",    IntegerType(), False),
    StructField("quantidade",    IntegerType(), False),
    StructField("preco_unitario",DoubleType(),  False),
])

df_clientes = spark.createDataFrame(clientes_data, schema=schema_clientes)
df_produtos  = spark.createDataFrame(produtos_data,  schema=schema_produtos)
df_pedidos   = spark.createDataFrame(pedidos_data,   schema=schema_pedidos)
df_itens     = spark.createDataFrame(itens_data,     schema=schema_itens)

# ─── INSERT via writeTo (API Iceberg) ───────────────────
df_clientes.writeTo("local.techstore.clientes").append()
print(f"[OK] clientes: {df_clientes.count()} registros inseridos")

df_produtos.writeTo("local.techstore.produtos").append()
print(f"[OK] produtos: {df_produtos.count()} registros inseridos")

df_pedidos.writeTo("local.techstore.pedidos").append()
print(f"[OK] pedidos: {df_pedidos.count()} registros inseridos")

df_itens.writeTo("local.techstore.itens_pedido").append()
print(f"[OK] itens_pedido: {df_itens.count()} registros inseridos")

# ─── Verificacao da leitura ─────────────────────────────
print("\n=" * 50)
print("CLIENTES (primeiros 5):")
spark.table("local.techstore.clientes").show(5)

print("=" * 50)
print("PRODUTOS (primeiros 5):")
spark.table("local.techstore.produtos").show(5, truncate=False)

print("=" * 50)
print("PEDIDOS por status:")
spark.table("local.techstore.pedidos").groupBy("status").count().orderBy("status").show()

---
## 6. UPDATE — Atualizando Registros via SQL

In [ ]:
print("Estado ANTES do UPDATE:")
spark.sql("SELECT * FROM local.techstore.clientes WHERE cliente_id = 1").show()

# ─── UPDATE 1: Atualizar cidade de cliente ──────────────
print("\n[UPDATE 1] Atualizando cidade do cliente_id=1...")
spark.sql("""
    UPDATE local.techstore.clientes
    SET cidade = 'Tubarao', estado = 'SC'
    WHERE cliente_id = 1
""")

# ─── UPDATE 2: Reajustar precos de Smartphones ──────────
print("[UPDATE 2] Reajustando preco dos Smartphones em 10%...")
spark.sql("""
    UPDATE local.techstore.produtos
    SET preco = ROUND(preco * 1.10, 2)
    WHERE categoria = 'Smartphones'
""")

# ─── UPDATE 3: Atualizar status de pedidos antigos ──────
print("[UPDATE 3] Marcando pedidos ENVIADO->ENTREGUE (data < 2024-06-01)...")
spark.sql("""
    UPDATE local.techstore.pedidos
    SET status = 'ENTREGUE'
    WHERE status = 'ENVIADO'
    AND data_pedido < '2024-06-01'
""")

print("\nEstado APOS os UPDATEs:")
print("\nCliente 1 atualizado:")
spark.sql("SELECT * FROM local.techstore.clientes WHERE cliente_id = 1").show()

print("\nSmartphones com novo preco:")
spark.sql("""
    SELECT * FROM local.techstore.produtos
    WHERE categoria = 'Smartphones'
""").show(truncate=False)

print("\nContagem de status nos pedidos:")
spark.sql("""
    SELECT status, COUNT(*) AS total
    FROM local.techstore.pedidos
    GROUP BY status
    ORDER BY status
""").show()

---
## 7. DELETE — Removendo Registros

In [ ]:
print("Contagem ANTES do DELETE:")
print(f"  Pedidos      : {spark.table('local.techstore.pedidos').count()}")
print(f"  Itens pedido : {spark.table('local.techstore.itens_pedido').count()}")
print(f"  Clientes     : {spark.table('local.techstore.clientes').count()}")

# Identificar pedidos cancelados
cancelados_df = spark.sql("""
    SELECT pedido_id FROM local.techstore.pedidos WHERE status = 'CANCELADO'
""")
ids_cancelados = [r.pedido_id for r in cancelados_df.collect()]
print(f"\n[DELETE 1] Removendo {len(ids_cancelados)} pedidos CANCELADOS...")

# ─── DELETE 1: Pedidos cancelados ───────────────────────
spark.sql("""
    DELETE FROM local.techstore.pedidos
    WHERE status = 'CANCELADO'
""")

# ─── DELETE 2: Itens dos pedidos cancelados ─────────────
print("[DELETE 2] Removendo itens dos pedidos cancelados...")
if ids_cancelados:
    ids_str = ", ".join(str(i) for i in ids_cancelados)
    spark.sql(f"""
        DELETE FROM local.techstore.itens_pedido
        WHERE pedido_id IN ({ids_str})
    """)

# ─── DELETE 3: Remover cliente por LGPD ─────────────────
print("[DELETE 3] Removendo cliente_id=50 (solicitacao LGPD)...")
spark.sql("""
    DELETE FROM local.techstore.clientes
    WHERE cliente_id = 50
""")

print("\nContagem APOS o DELETE:")
print(f"  Pedidos      : {spark.table('local.techstore.pedidos').count()}")
print(f"  Itens pedido : {spark.table('local.techstore.itens_pedido').count()}")
print(f"  Clientes     : {spark.table('local.techstore.clientes').count()} (era 100)")

---
## 8. MERGE — Upsert com MERGE INTO

In [ ]:
# Criar view temporaria com novos dados
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW novos_clientes AS
    SELECT * FROM VALUES
        (1,   'Alice Souza ATUALIZADA',  'alice@techstore.com',   'Criciuma',  'SC', DATE('2024-01-05')),
        (3,   'Carlos NOVO EMAIL',        'carlos3@techstore.com', 'Sao Paulo', 'SP', DATE('2022-05-20')),
        (101, 'Novo Cliente 101',          'novo101@techstore.com', 'Goiania',   'GO', DATE('2025-01-02')),
        (102, 'Novo Cliente 102',          'novo102@techstore.com', 'Salvador',  'BA', DATE('2025-01-03')),
        (103, 'Novo Cliente 103',          'novo103@techstore.com', 'Recife',    'PE', DATE('2025-01-04'))
    AS t(cliente_id, nome, email, cidade, estado, data_cadastro)
""")

print("Dados para MERGE:")
spark.sql("SELECT * FROM novos_clientes").show()

# ─── MERGE INTO ─────────────────────────────────────────
spark.sql("""
    MERGE INTO local.techstore.clientes AS destino
    USING novos_clientes AS fonte
    ON destino.cliente_id = fonte.cliente_id
    WHEN MATCHED THEN
        UPDATE SET
            destino.nome         = fonte.nome,
            destino.cidade       = fonte.cidade,
            destino.estado       = fonte.estado
    WHEN NOT MATCHED THEN
        INSERT (cliente_id, nome, email, cidade, estado, data_cadastro)
        VALUES (
            fonte.cliente_id, fonte.nome, fonte.email,
            fonte.cidade, fonte.estado, fonte.data_cadastro
        )
""")

print("\nResultado apos MERGE:")
total = spark.table("local.techstore.clientes").count()
print(f"Total de clientes: {total} (era 99 apos DELETE, agora +3 novos)")

print("\nClientes atualizados (ids 1 e 3):")
spark.sql("""
    SELECT * FROM local.techstore.clientes
    WHERE cliente_id IN (1, 3)
    ORDER BY cliente_id
""").show()

print("\nNovos clientes inseridos (ids 101-103):")
spark.sql("""
    SELECT * FROM local.techstore.clientes
    WHERE cliente_id >= 101
    ORDER BY cliente_id
""").show()

---
## 9. Time Travel — Consultando Snapshots Históricos

O Iceberg mantém um histórico de **snapshots** — cada operação de escrita gera um novo snapshot imutável.

In [ ]:
print("=" * 60)
print("HISTORICO DE SNAPSHOTS — tabela clientes")
print("=" * 60)
snapshots_df = spark.sql("""
    SELECT
        snapshot_id,
        committed_at,
        operation,
        summary
    FROM local.techstore.clientes.snapshots
    ORDER BY committed_at
""")
snapshots_df.show(truncate=False)

# Capturar o snapshot_id inicial (1o snapshot = INSERT original)
snapshot_inicial = snapshots_df.orderBy("committed_at").first()
snapshot_id_v0   = snapshot_inicial.snapshot_id

print(f"Snapshot inicial: {snapshot_id_v0}")
print(f"Operacao         : {snapshot_inicial.operation}")

# ─── Time Travel por snapshot_id ─────────────────────────
print("\nTime Travel - snapshot inicial (apos INSERT):")
spark.sql(f"""
    SELECT cliente_id, nome, cidade, estado
    FROM local.techstore.clientes
    FOR SYSTEM_VERSION AS OF {snapshot_id_v0}
    WHERE cliente_id <= 5
    ORDER BY cliente_id
""").show()

print("Versao atual (apos UPDATE/DELETE/MERGE):")
spark.sql("""
    SELECT cliente_id, nome, cidade, estado
    FROM local.techstore.clientes
    WHERE cliente_id <= 5
    ORDER BY cliente_id
""").show()

# ─── Comparacao: total de registros por versao ────────────
print("Comparacao de totais:")
total_v0 = spark.sql(f"""
    SELECT COUNT(*) AS total FROM local.techstore.clientes
    FOR SYSTEM_VERSION AS OF {snapshot_id_v0}
""").first().total
total_atual = spark.table("local.techstore.clientes").count()
print(f"  Snapshot inicial ({snapshot_id_v0}): {total_v0} registros")
print(f"  Versao atual                       : {total_atual} registros")

---
## 10. Análises com Spark SQL

In [ ]:
print("=" * 60)
print("TOP 10 CLIENTES POR RECEITA TOTAL")
print("=" * 60)
spark.sql("""
    SELECT
        c.nome,
        c.cidade,
        c.estado,
        COUNT(p.pedido_id)          AS total_pedidos,
        ROUND(SUM(p.valor_total),2) AS receita_total
    FROM local.techstore.pedidos p
    JOIN local.techstore.clientes c ON p.cliente_id = c.cliente_id
    GROUP BY c.nome, c.cidade, c.estado
    ORDER BY receita_total DESC
    LIMIT 10
""").show(truncate=False)

print("=" * 60)
print("RECEITA POR CATEGORIA DE PRODUTO")
print("=" * 60)
spark.sql("""
    SELECT
        pr.categoria,
        COUNT(DISTINCT pr.produto_id) AS qtd_produtos,
        SUM(i.quantidade)             AS unidades_vendidas,
        ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS receita
    FROM local.techstore.itens_pedido i
    JOIN local.techstore.produtos pr ON i.produto_id = pr.produto_id
    GROUP BY pr.categoria
    ORDER BY receita DESC
""").show()

print("=" * 60)
print("DISTRIBUICAO DE CLIENTES POR ESTADO")
print("=" * 60)
spark.sql("""
    SELECT
        estado,
        COUNT(*) AS total_clientes
    FROM local.techstore.clientes
    GROUP BY estado
    ORDER BY total_clientes DESC
""").show()

---
## 11. Manutenção de Tabelas Iceberg

In [ ]:
# ─── Listar arquivos de dados da tabela ──────────────────
print("=" * 60)
print("ARQUIVOS DE DADOS — tabela clientes")
print("=" * 60)
spark.sql("""
    SELECT
        file_path,
        file_format,
        record_count,
        file_size_in_bytes
    FROM local.techstore.clientes.files
""").show(truncate=False)

# ─── Listar partitions ───────────────────────────────────
print("=" * 60)
print("PARTICOES — tabela clientes (por estado)")
print("=" * 60)
spark.sql("""
    SELECT * FROM local.techstore.clientes.partitions
    ORDER BY spec_id, partition
""").show()

# ─── Compactar arquivos pequenos ─────────────────────────
print("=" * 60)
print("COMPACTACAO de arquivos pequenos (rewrite_data_files)")
print("=" * 60)
try:
    resultado = spark.sql("""
        CALL local.system.rewrite_data_files(
            table => 'local.techstore.clientes'
        )
    """)
    resultado.show()
    print("Compactacao concluida.")
except Exception as e:
    print(f"Compactacao ignorada (poucos arquivos): {e}")

print("\nTabelas do namespace techstore:")
spark.sql("SHOW TABLES IN local.techstore").show()

---
## 12. Conclusão

### O que foi demonstrado neste notebook:

| Operação | Método utilizado | Resultado |
|----------|-----------------|----------|
| **INSERT** | `df.writeTo().append()` | 4 tabelas criadas |
| **UPDATE** | `spark.sql("UPDATE ...")` | Cidades, preços e status atualizados |
| **DELETE** | `spark.sql("DELETE FROM ...")` | Pedidos cancelados e cliente removido |
| **MERGE** | `spark.sql("MERGE INTO ...")` | 2 atualizados + 3 inseridos |
| **Time Travel** | `FOR SYSTEM_VERSION AS OF` | Consulta de snapshot histórico |
| **Snapshots** | `.snapshots` metadata table | Histórico completo de operações |
| **Manutenção** | `rewrite_data_files` | Compactação de arquivos pequenos |

### Principais vantagens do Apache Iceberg:
- **Especificação aberta** — qualquer engine pode ler (Spark, Trino, Flink, Hive)
- **Partition Evolution** — mudar particionamento sem reescrever dados
- **Time Travel** por snapshot — histórico imutável e auditável
- **Escala massiva** — metadados em 3 camadas para tabelas petabyte
- **SQL padrão** — UPDATE, DELETE, MERGE nativos e portáveis

In [ ]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")